# Day 51：大模型微调理论、决策树（Prompt vs RAG vs FT）与 GPU 显存预算计算器

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》第五阶段（Stage 5）第一课——**Day 51**！在企业私有化垂类大模型交付前，我们必须用工程化工具回答两个核心问题：**“该用 Prompt、RAG、LoRA 还是 QLoRA？”**以及**“训练到底需要多少 GB 显存、选什么规格的 GPU？”**本 Notebook 将带你实战 `project_iterations/day_51` 中的 `vram_calculator` 模块（`v0.5.0-day51`）。

## 1. 环境初始化与导入 `project_iterations/day_51` 模块

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day51_dir = repo_root / "project_iterations" / "day_51"
if str(day51_dir) not in sys.path:
    sys.path.insert(0, str(day51_dir))

from zhishu_agent import (
    estimate_training_vram_gb,
    recommend_adaptation_strategy,
    recommend_gpu,
)

print("已加载模块路径:", day51_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_51


## 2. 实战演练一：Prompt vs RAG vs LoRA vs QLoRA 决策树
测试企业常见四类场景，观察 `recommend_adaptation_strategy` 如何结合实时知识、风格格式、标注数据与延迟 SLA 做出工程决策。

In [2]:
cases = [
    ("HR 最新制度问答", True, False, False, 1500),
    ("金融研报 JSON 格式微调", False, True, True, 1200),
    ("毫秒级风控标签分类", False, True, True, 300),
    ("实时投研 + 固定研报模板", True, True, True, 1000),
    ("通用办公邮件润色", False, False, False, 2000),
]

for title, fresh, style, labeled, budget in cases:
    rec = recommend_adaptation_strategy(fresh, style, labeled, budget)
    print(f"[{title}] -> 策略: {rec.strategy:6s} | RAG联动: {str(rec.hybrid_with_rag):5s} | {rec.rationale}")

[HR 最新制度问答] -> 策略: RAG    | RAG联动: True  | 业务核心诉求为高频动态知识注入，无需改变基座模型内在表达风格。微调无法解决实时知识遗忘与更新成本问题，应优先采用 RAG 外挂检索增强。
[金融研报 JSON 格式微调] -> 策略: QLoRA  | RAG联动: False | 已具备标注数据且延迟预算充裕（1200ms >= 500ms）。QLoRA（4-bit NF4 基座 + LoRA 适配器 + Paged AdamW）可降低约 70% 训练显存，是企业私有化垂类微调性价比最高的首选方案。
[毫秒级风控标签分类] -> 策略: LoRA   | RAG联动: False | 已具备标注数据且延迟预算严格（300ms < 500ms）。采用 LoRA-FP16 微调后可将旁路权重无损合并（merge_and_unload）回 FP16 基座，避免 4-bit 反量化推理延迟开销。
[实时投研 + 固定研报模板] -> 策略: QLoRA  | RAG联动: True  | 已具备标注数据且延迟预算充裕（1000ms >= 500ms）。QLoRA（4-bit NF4 基座 + LoRA 适配器 + Paged AdamW）可降低约 70% 训练显存，是企业私有化垂类微调性价比最高的首选方案。建议搭配 RAG 注入实时外部知识（RAG + 微调混合架构）。
[通用办公邮件润色] -> 策略: Prompt | RAG联动: False | 通用能力即可满足需求，无需实时知识注入或特定格式微调。优先通过系统提示词工程（System Prompt + CoT + Few-Shot）实现最快交付与最低维护成本。


## 3. 实战演练二：Qwen2.5-7B / 14B / 32B 训练显存拆解与 GPU 选型
对比 `Full-FP16`、`LoRA-FP16` 与 `QLoRA-4bit` 在不同参数量与序列长度下的显存占用与推荐显卡。

In [3]:
configs = [
    ("Qwen2.5-7B", 7.0, "Full-FP16", 1, 2048),
    ("Qwen2.5-7B", 7.0, "LoRA-FP16", 1, 2048),
    ("Qwen2.5-7B", 7.0, "QLoRA-4bit", 1, 2048),
    ("Qwen2.5-7B", 7.0, "LoRA-FP16", 4, 4096),
    ("Qwen2.5-14B", 14.0, "QLoRA-4bit", 1, 2048),
    ("Qwen2.5-32B", 32.0, "QLoRA-4bit", 1, 2048),
]

for name, params_b, method, bs, seq in configs:
    v = estimate_training_vram_gb(params_b, method=method, batch_size=bs, seq_len=seq, lora_rank=16)
    g = recommend_gpu(v.total_gb)
    print(
        f"{name:11s} | {v.method:10s} | bs={bs}, seq={seq:4d} | "
        f"W={v.weights_gb:5.2f}G, G={v.gradients_gb:5.2f}G, Opt={v.optimizer_gb:5.2f}G, "
        f"Act={v.activations_gb:5.2f}G -> Total={v.total_gb:6.2f}G | {g.tier} x{g.gpu_count}"
    )

Qwen2.5-7B  | Full-FP16  | bs=1, seq=2048 | W=13.04G, G=13.04G, Opt=78.23G, Act= 3.66G -> Total=108.97G | Multi-GPU x2
Qwen2.5-7B  | LoRA-FP16  | bs=1, seq=2048 | W=13.14G, G= 0.10G, Opt= 0.63G, Act= 2.44G -> Total= 17.31G | RTX 4090 24G x1
Qwen2.5-7B  | QLoRA-4bit | bs=1, seq=2048 | W= 3.69G, G= 0.10G, Opt= 0.31G, Act= 2.44G -> Total=  7.55G | RTX 4090 24G x1
Qwen2.5-7B  | LoRA-FP16  | bs=4, seq=4096 | W=13.14G, G= 0.10G, Opt= 0.63G, Act=19.52G -> Total= 34.39G | A100 40G x1
Qwen2.5-14B | QLoRA-4bit | bs=1, seq=2048 | W= 7.38G, G= 0.21G, Opt= 0.63G, Act= 3.15G -> Total= 12.36G | RTX 4090 24G x1
Qwen2.5-32B | QLoRA-4bit | bs=1, seq=2048 | W=16.87G, G= 0.48G, Opt= 1.43G, Act= 4.53G -> Total= 24.31G | A100 40G x1


## 4. 课后作业自动化校验
对 Day 51 三道课后作业的核心断言进行端到端验证。

In [4]:
assert recommend_adaptation_strategy(True, False, False, 1500).strategy == "RAG"
assert recommend_adaptation_strategy(False, True, True, 1200).strategy == "QLoRA"
assert recommend_adaptation_strategy(False, True, True, 300).strategy == "LoRA"

v_qlora = estimate_training_vram_gb(7.0, method="QLoRA-4bit", batch_size=1, seq_len=2048)
v_lora_big = estimate_training_vram_gb(7.0, method="LoRA-FP16", batch_size=4, seq_len=4096)
v_full = estimate_training_vram_gb(7.0, method="Full-FP16", batch_size=1, seq_len=2048)

assert recommend_gpu(v_qlora.total_gb).tier == "RTX 4090 24G"
assert recommend_gpu(v_lora_big.total_gb).tier == "A100 40G"
assert recommend_gpu(v_full.total_gb).tier == "Multi-GPU"

print("Day 51 所有课堂实战与课后作业校验 100% 通过！")

Day 51 所有课堂实战与课后作业校验 100% 通过！
